In [2]:
# ============================================================
# CELL 1: Install dependencies & setup Week-4 directory
# ============================================================
!pip install -q astropy photutils huggingface_hub tqdm

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from tqdm.auto import tqdm
from astropy.stats import SigmaClip
from astropy.convolution import convolve
from astropy.visualization import make_lupton_rgb
from photutils.background import Background2D, MedianBackground
from photutils.segmentation import SourceFinder, make_2dgaussian_kernel
from huggingface_hub import hf_hub_download

# Uncomment below to save directly to Google Drive (recommended for Colab)
# from google.colab import drive
# drive.mount('/content/drive')
# BASE = Path('/content/drive/MyDrive/Week-4')

BASE = Path("Week-4")
for sub in [
    "data/cutouts",
    "results/pseudo_labels",
    "results/predictions",
    "results/metrics",
    "results/checkpoints",
]:
    (BASE / sub).mkdir(parents=True, exist_ok=True)

print("Workspace:", BASE.resolve())

Workspace: /content/Week-4


In [6]:
# ============================================================
# CELL 2: Normalization (Task A) & Pseudo-Label Pipeline (Task B)
# ============================================================
def normalize_and_rgb(griz: np.ndarray):
    """Background-subtract g, r, z with Background2D, divide by sky RMS (S/N), convert to Lupton RGB."""
    sigma_clip = SigmaClip(sigma=3.0)
    bkg_est = MedianBackground()

    grz = np.nan_to_num(griz[[0, 1, 3]].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    grz_sn = np.zeros_like(grz)

    for i in range(3):
        bkg = Background2D(grz[i], box_size=(32, 32), filter_size=(3, 3),
                           sigma_clip=sigma_clip, bkg_estimator=bkg_est)
        grz_sn[i] = (grz[i] - bkg.background) / np.maximum(bkg.background_rms, 1e-6)

    rgb = make_lupton_rgb(grz_sn[2], grz_sn[1], grz_sn[0], Q=8, stretch=2.5).astype(np.float32) / 255.0
    return grz_sn, rgb


def make_pseudo_label(grz_sn: np.ndarray, threshold_sigma: float = 1.5, npixels: int = 10):
    """Detect & deblend sources on smoothed g+r+z S/N image -> 3-class 160x160 mask."""
    det = grz_sn.sum(axis=0)
    H, W = det.shape
    cy, cx = H // 2, W // 2

    bkg = Background2D(det, box_size=(32, 32), filter_size=(3, 3),
                       sigma_clip=SigmaClip(sigma=3.0), bkg_estimator=MedianBackground())
    conv = convolve(det - bkg.background, make_2dgaussian_kernel(fwhm=3.0, size=5))

    finder = SourceFinder(n_pixels=npixels, deblend=True, n_levels=32, contrast=0.001)
    segmap = finder(conv, threshold_sigma * bkg.background_rms)

    mask = np.zeros((H, W), dtype=np.uint8)
    if segmap is None:
        return np.zeros((H, W), dtype=np.int32), mask, {"target_id": 0, "dist": 999.0, "target_frac": 0.0, "other_frac": 0.0}

    seg = segmap.data
    target_id = int(seg[cy, cx])
    dist = 0.0

    if target_id == 0:
        labels = np.unique(seg)
        labels = labels[labels > 0]
        if len(labels) > 0:
            best_d, best_id = 1e9, 0
            for lbl in labels:
                ys, xs = np.where(seg == lbl)
                d = np.hypot(ys.mean() - cy, xs.mean() - cx)
                if d < best_d:
                    best_d, best_id = d, int(lbl)
            target_id, dist = best_id, float(best_d)

    if target_id > 0:
        mask[seg == target_id] = 1
        mask[(seg > 0) & (seg != target_id)] = 2

    info = {
        "target_id": target_id,
        "dist": round(dist, 2),
        "target_frac": float((mask == 1).mean()),
        "other_frac": float((mask == 2).mean()),
    }
    return seg, mask, info

In [7]:
# ============================================================
# CELL 3: Download 3,000 DESI Cutouts, Generate Masks & splits.json
# ============================================================
DATASET = "MultimodalUniverse/legacysurvey"
REVISION = "112bddb46da031de2807ced8fe28fb79ed13b515"
N_TOTAL = 3000

records = []
meta_stats = {}

for shard in range(6):
    if len(records) >= N_TOTAL:
        break
    pq_path = hf_hub_download(
        DATASET, f"dr10_south_21/train-{shard:05d}-of-00165.parquet",
        repo_type="dataset", revision=REVISION,
    )
    df = pd.read_parquet(pq_path, columns=["object_id", "image"])

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Shard {shard}"):
        if len(records) >= N_TOTAL:
            break
        key = str(row["object_id"])
        flux = np.stack([np.stack(band) for band in row["image"]["flux"]]).astype(np.float32)
        if flux.shape != (4, 160, 160):
            continue

        grz_sn, rgb = normalize_and_rgb(flux)
        _, mask, info = make_pseudo_label(grz_sn)

        np.savez_compressed(BASE / f"data/cutouts/{key}.npz", rgb=rgb, grz_sn=grz_sn.astype(np.float16))
        np.save(BASE / f"results/pseudo_labels/{key}.npy", mask)

        records.append(key)
        meta_stats[key] = info

rng = np.random.default_rng(42)
shuffled = rng.permutation(records).tolist()
n_tr, n_va = int(0.70 * len(shuffled)), int(0.10 * len(shuffled))

splits = {
    "train": shuffled[:n_tr],
    "validation": shuffled[n_tr : n_tr + n_va],
    "test": shuffled[n_tr + n_va :],
}
(BASE / "data/splits.json").write_text(json.dumps(splits, indent=2))
(BASE / "data/mask_stats.json").write_text(json.dumps(meta_stats, indent=2))
print("Splits saved:", {k: len(v) for k, v in splits.items()})

Shard 0:   0%|          | 0/549 [00:00<?, ?it/s]

Shard 1:   0%|          | 0/549 [00:00<?, ?it/s]

dr10_south_21/train-00002-of-00165.parqu(…): reconstructing file:   0%|          |  0.00B /  391MB            

dr10_south_21/train-00002-of-00165.parqu(…): downloading bytes:           |  0.00B            

Shard 2:   0%|          | 0/549 [00:00<?, ?it/s]

dr10_south_21/train-00003-of-00165.parqu(…): reconstructing file:   0%|          |  0.00B /  392MB            

... (5 outputs left out)

dr10_south_21/train-00005-of-00165.parqu(…): reconstructing file:   0%|          |  0.00B /  390MB            

dr10_south_21/train-00005-of-00165.parqu(…): downloading bytes:           |  0.00B            

Shard 5:   0%|          | 0/549 [00:00<?, ?it/s]

Splits saved: {'train': 2100, 'validation': 300, 'test': 600}


In [8]:
# ============================================================
# CELL 4: QA Exclusions, Hard-Set Selection & 50-Mask Visual Review
# ============================================================
excluded_ids = [
    k for k, s in meta_stats.items()
    if s["target_id"] == 0 or s["dist"] > 25.0 or s["target_frac"] > 0.60
]
(BASE / "data/excluded_ids.json").write_text(json.dumps(excluded_ids, indent=2))
print(f"Auto-excluded broken masks: {len(excluded_ids)}")

# Class pixel distribution on train
counts = np.zeros(3, dtype=np.int64)
for k in splits["train"]:
    if k not in excluded_ids:
        m = np.load(BASE / f"results/pseudo_labels/{k}.npy")
        for c in range(3):
            counts[c] += (m == c).sum()
fracs = counts / counts.sum()
print(f"Train pixels -> Bg: {fracs[0]:.2%}, Target: {fracs[1]:.2%}, Other: {fracs[2]:.2%}")

# Pick 20 hard test galaxies (valid target + heavy crowding/blending)
test_cands = [
    (k, meta_stats[k]["other_frac"])
    for k in splits["test"]
    if k not in excluded_ids and meta_stats[k]["target_frac"] >= 0.015
]
test_cands.sort(key=lambda x: x[1], reverse=True)
hard_ids = [k for k, _ in test_cands[:20]]
(BASE / "data/hard_set.json").write_text(json.dumps(hard_ids, indent=2))
print("Saved 20 hard test IDs to data/hard_set.json")

# Visual review of 50 random training masks (5 pages x 10 rows)
cmap = ListedColormap(["black", "gold", "royalblue"])
sample_ids = rng.choice([k for k in splits["train"] if k not in excluded_ids], 50, replace=False)

for page in range(5):
    fig, axes = plt.subplots(10, 3, figsize=(9, 24))
    for row, key in enumerate(sample_ids[page * 10 : (page + 1) * 10]):
        d = np.load(BASE / f"data/cutouts/{key}.npz")
        seg, mask, _ = make_pseudo_label(d["grz_sn"].astype(np.float32))
        axes[row, 0].imshow(d["rgb"]); axes[row, 0].set_title(key, fontsize=8)
        axes[row, 1].imshow(seg, cmap="nipy_spectral"); axes[row, 1].set_title("Segments", fontsize=8)
        axes[row, 2].imshow(mask, cmap=cmap, vmin=0, vmax=2); axes[row, 2].set_title("3-Class Mask", fontsize=8)
        for ax in axes[row]:
            ax.axis("off")
    fig.tight_layout()
    fig.savefig(BASE / f"results/pseudo_labels/review_page_{page + 1}.png", dpi=110)
    plt.close(fig)

print("Saved 50-mask review pages to results/pseudo_labels/")

Auto-excluded broken masks: 1
Train pixels -> Bg: 95.06%, Target: 1.25%, Other: 3.69%
Saved 20 hard test IDs to data/hard_set.json
Saved 50-mask review pages to results/pseudo_labels/


In [9]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/Week-4 /content/drive/MyDrive/Week-4
print("Backed up to Google Drive!")

Mounted at /content/drive
Backed up to Google Drive!
